# Stage 11: Benchmark Comparison

This notebook creates a formal benchmark-aware comparison across Equal Weight, Inverse Volatility, HRP, and HERC.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
if project_root.name == "11_benchmark_comparison":
    project_root = project_root.parents[1]

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.benchmarks import (
    build_performance_comparison_table,
    compute_relative_performance,
    run_strategy_comparison,
)
from src.dashboard.plots import (
    plot_drawdown_curves,
    plot_final_value_comparison,
    plot_metric_comparison,
    plot_performance_curves,
    plot_relative_performance,
)


## 1. Load Data

In [ ]:
rng = np.random.default_rng(2042)
dates = pd.date_range(start="2021-01-01", periods=504, freq="B")
common_factor = rng.normal(0.0002, 0.0045, size=(len(dates), 1))
idiosyncratic = rng.normal(
    loc=0.0003,
    scale=np.array([0.009, 0.012, 0.015, 0.010, 0.013]),
    size=(len(dates), 5),
)
returns_df = pd.DataFrame(
    common_factor + idiosyncratic,
    index=dates,
    columns=["Equity", "IT", "Gold", "Bonds", "Energy"],
)
returns_df.head()


## 2. Generate Returns

In [ ]:
returns_df.describe().T[['mean', 'std']]


## 3. Run All Strategies

In [ ]:
strategy_results = run_strategy_comparison(
    returns_df,
    strategy_names=["Equal Weight", "Inverse Volatility", "HRP", "HERC"],
    covariance_method="ledoit_wolf",
    train_window=252,
    rebalance_frequency="M",
    initial_capital=1_000_000.0,
)
list(strategy_results.keys())


## 4. Compare Performance Metrics

In [ ]:
performance_comparison_df = build_performance_comparison_table(strategy_results)
performance_comparison_df.round(4)


## 5. Compare Drawdowns

In [ ]:
drawdown_curves = {
    strategy_name: result['drawdown']
    for strategy_name, result in strategy_results.items()
}
plot_drawdown_curves(drawdown_curves).show()


## 6. Compare Final Values

In [ ]:
growth_curves = {
    strategy_name: result['portfolio_values']
    for strategy_name, result in strategy_results.items()
}
plot_performance_curves(growth_curves).show()
plot_final_value_comparison(performance_comparison_df).show()


## 7. Compute Relative Performance Versus Equal Weight

In [ ]:
relative_performance_df = compute_relative_performance(
    performance_comparison_df,
    benchmark_name='Equal Weight',
)
relative_performance_df.round(4)


In [ ]:
plot_metric_comparison(performance_comparison_df, 'sharpe').show()
plot_relative_performance(relative_performance_df, 'excess_cagr').show()


## 8. Interpret Results

In [ ]:
best_sharpe = performance_comparison_df['sharpe'].idxmax()
best_calmar = performance_comparison_df['calmar'].idxmax()
most_defensive = performance_comparison_df['max_drawdown'].idxmax()
relative_to_equal = relative_performance_df.set_index('strategy')

discussion = [
    '### Research Questions',
    f'- Best Sharpe: `{best_sharpe}`',
    f'- Best Calmar: `{best_calmar}`',
    f'- Most defensive on max drawdown: `{most_defensive}`',
    f"- HRP excess CAGR vs Equal Weight: `{relative_to_equal.loc['HRP', 'excess_cagr']:.4f}`",
    f"- HERC excess CAGR vs Equal Weight: `{relative_to_equal.loc['HERC', 'excess_cagr']:.4f}`",
    f"- HERC drawdown difference vs Equal Weight: `{relative_to_equal.loc['HERC', 'drawdown_difference']:.4f}`",
    '- This framework answers `better than what?` by forcing every strategy comparison through an explicit benchmark baseline.',
]

display(Markdown('\n'.join(discussion)))
